In [13]:
import pandas as pd
import seaborn
import matplotlib.pyplot as plt
import numpy as np
import streamlit as sl

In [14]:
path = "Tabela_13_Meio_de_transporte.xlsx"

In [15]:
df = pd.read_excel(path, skiprows=1, sheet_name=1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20,10))
cores = ['#ff9999', '#66b3ff', '#99ff99', '#ffcc99', '#c2c2f0', '#ffb3e6']

grupos = ['Branca', 'Preta ou parda', 'Indígena']

for i, col in enumerate(grupos):
  axes[i].pie(
      df[col],
      labels=df['Meio de transporte/cor ou raça'],
      autopct='%1.3f%%',  # COLOQUE EM 3 CASAS DECIMAIS 
      startangle=180,  # Ângulo inicial de rotação
      colors=cores,
      wedgeprops={'edgecolor': 'white', 'linewidth': 1.2},
  )
  axes[i].set_title(f'Cor/Raça: {col}', fontsize=12, fontweight='bold')

plt.suptitle(
    'Desigualdade nos meio de transporte',
    fontsize=14,
    fontweight='bold',
)
plt.tight_layout()
plt.savefig('grafico_transporte.png', dpi=300)
plt.show()

### Diferença Transporte coletivo:
- Preto **34.6%** - Branca 25.2% - Indígena 22.4%

### Diferença Automóvel, taxi ou assemelhados:
- Preto 20.6% - Branca **25.2%** - Indígena 15.2%

### Diferença Motocicleta ou Mototaxi:
- Preto 14.2% - Branca 9.9% - Indígena **15.4%**

### Diferença Bicicleta:
- Preto 5.2% - Branca 3.2% - Indígena **5.9%**

### Diferença A pé:
- Preto 24.8% - Branca 19.4% - Indígena **37.5%**

### Diferença Outros:
- Preto 0.6% - Branca 0.5% - Indígena **3.60%**

In [ ]:
df = pd.read_excel(path, sheet_name=0, skiprows=2, header=None)

# Definir nomes baseados na estrutura (Raça_Meio)
colunas = ['Local', 
           'Branca_A pé', 'Branca_Bicicleta', 'Branca_Motocicleta', 'Branca_Carro', 'Branca_Coletivo', 'Branca_Outros',
           'PretaParda_A pé', 'PretaParda_Bicicleta', 'PretaParda_Motocicleta', 'PretaParda_Carro', 'PretaParda_Coletivo', 'PretaParda_Outros',
           'Indigena_A pé', 'Indigena_Bicicleta', 'Indigena_Motocicleta', 'Indigena_Carro', 'Indigena_Coletivo', 'Indigena_Outros']
df = df.iloc[:, :19]
df.columns = colunas
df['Local'] = df['Local'].astype(str).str.strip()

# 2. Formato Longo (melt)
df_longo = df.melt(
    id_vars=['Local'],
    value_vars=colunas[1:],
    var_name="Categoria",
    value_name="Percentual"
)

# Limpeza e separação de Raça/Transporte
df_longo['Percentual'] = pd.to_numeric(df_longo['Percentual'].replace(['-', ' ', '.', 'nan'], np.nan), errors='coerce').fillna(0)
df_longo[['Raca', 'Transporte']] = df_longo['Categoria'].str.split('_', expand=True)

# Média geral por local e tipo de transporte
df_geral = df_longo.groupby(['Local', 'Transporte'])['Percentual'].mean().reset_index()

# 3. Listar estados e cidades
estados = ['Rondônia', 'Acre', 'Amazonas', 'Roraima', 'Pará', 'Amapá', 'Tocantins', 'Maranhão', 'Piauí', 'Ceará', 'Rio Grande do Norte', 'Paraíba', 'Pernambuco', 'Alagoas', 'Sergipe', 'Bahia', 'Minas Gerais', 'Espírito Santo', 'Rio de Janeiro', 'São Paulo', 'Paraná', 'Santa Catarina', 'Rio Grande do Sul', 'Mato Grosso do Sul', 'Mato Grosso', 'Goiás', 'Distrito Federal']
excluir = estados + ['Brasil', 'Norte', 'Nordeste', 'Sudeste', 'Sul', 'Centro-oeste', 'nan']

df_est = df_geral[df_geral['Local'].isin(estados)]
df_cid = df_geral[~df_geral['Local'].isin(excluir)]

# Respostas
coletivo = df_est[df_est['Transporte'] == 'Coletivo']
print("Mais Coletivo (Estado):", coletivo.loc[coletivo['Percentual'].idxmax(), 'Local'])
print("Menos Coletivo (Estado):", coletivo.loc[coletivo['Percentual'].idxmin(), 'Local'])

bike = df_cid[df_cid['Transporte'] == 'Bicicleta']
print("Mais Bicicleta (Cidade):", bike.loc[bike['Percentual'].idxmax(), 'Local'])

carro = df_cid[df_cid['Transporte'] == 'Carro']
print("Mais Carro (Cidade):", carro.loc[carro['Percentual'].idxmax(), 'Local'])